In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [7]:
df = pd.read_csv("../../data/rolling_feature_engineered_tier_1_games.csv", sep=",")
df.head()

,previous_10_game_team1_player1_average_kills,previous_10_game_team1_player1_average_deaths,previous_10_game_team1_player1_average_assists,previous_10_game_team1_player1_average_adr,previous_10_game_team1_player1_average_kast,previous_10_game_team1_player1_average_kddiff,previous_10_game_team1_player2_average_kills,previous_10_game_team1_player2_average_deaths,previous_10_game_team1_player2_average_assists,previous_10_game_team1_player2_average_adr,...,team1_player1_id,team1_player2_id,team1_player3_id,team1_player4_id,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id
0,16.000000,15.000000,4.000000,75.000000,70.000000,1.000000,16.000000,15.000000,4.000000,75.000000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
1,12.000000,17.000000,4.000000,59.100000,62.500000,-5.000000,13.000000,11.000000,2.000000,52.600000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
2,13.500000,16.500000,4.500000,66.250000,71.750000,-3.000000,18.500000,11.000000,4.000000,82.550000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
3,14.612903,14.645161,4.258065,69.832258,71.535484,-0.032258,14.612903,14.645161,4.258065,69.832258,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0
4,13.000000,5.000000,1.000000,81.300000,86.700000,8.000000,11.000000,6.000000,4.000000,72.100000,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0


In [10]:
target = "team1_win"

df[target].value_counts(normalize=True)

team1_win
1    0.548836
0    0.451164
Name: proportion, dtype: float64

sort time

In [11]:
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
print(df["datetime"].head())
print(df["datetime"].tail())

0   2023-10-27 11:00:00
1   2023-10-27 11:00:00
2   2023-10-27 11:00:00
3   2023-10-27 14:15:00
4   2023-10-27 14:15:00
Name: datetime, dtype: datetime64[ns]
5534   2026-03-28 19:40:00
5535   2026-03-28 19:40:00
5536   2026-03-28 19:40:00
5537   2026-03-29 13:30:00
5538   2026-03-29 13:30:00
Name: datetime, dtype: datetime64[ns]


Transform player-level data into team-level data.

In [12]:

stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

for stat in stats:
    team1_cols = [
        f"previous_10_game_team1_player{i}_average_{stat}"
        for i in range(1, 6)
    ]
    
    team2_cols = [
        f"previous_10_game_team2_player{i}_average_{stat}"
        for i in range(1, 6)
    ]
    
    df[f"team1_avg_{stat}"] = df[team1_cols].mean(axis=1)
    df[f"team2_avg_{stat}"] = df[team2_cols].mean(axis=1)
    
    df[f"{stat}_diff"] = df[f"team1_avg_{stat}"] - df[f"team2_avg_{stat}"]

# Check new features
new_features = []

for stat in stats:
    new_features.extend([
        f"team1_avg_{stat}",
        f"team2_avg_{stat}",
        f"{stat}_diff"
    ])

df[new_features].head()

,team1_avg_kills,team2_avg_kills,kills_diff,team1_avg_deaths,team2_avg_deaths,deaths_diff,team1_avg_assists,team2_avg_assists,assists_diff,team1_avg_adr,team2_avg_adr,adr_diff,team1_avg_kast,team2_avg_kast,kast_diff,team1_avg_kddiff,team2_avg_kddiff,kddiff_diff
0,16.000000,16.000000,0.0,15.000000,15.000000,0.0,4.000000,4.000000,0.0,75.000000,75.000000,0.00,70.000000,70.000000,0.00,1.000000,1.000000,0.0
1,15.600000,15.000000,0.6,15.000000,15.600000,-0.6,2.800000,6.000000,-3.2,68.160000,67.640000,0.52,68.320000,66.640000,1.68,0.600000,-0.600000,1.2
2,15.100000,13.600000,1.5,13.600000,15.200000,-1.6,3.800000,4.600000,-0.8,70.120000,67.010000,3.11,72.260000,63.790000,8.47,1.500000,-1.600000,3.1
3,14.612903,14.612903,0.0,14.645161,14.645161,0.0,4.258065,4.258065,0.0,69.832258,69.832258,0.00,71.535484,71.535484,0.00,-0.032258,-0.032258,0.0
4,13.600000,4.600000,9.0,4.600000,13.600000,-9.0,4.200000,1.200000,3.0,92.420000,45.620000,46.80,88.000000,40.000000,48.00,9.000000,-9.000000,18.0


The model compares the difference in average scores between team1 and team2 on the map in the past.

In [13]:

df["map_score_diff"] = (
    df["team1_previous_10_average_map_score"]
    - df["team2_previous_10_average_map_score"]
)

df[[
    "team1_previous_10_average_map_score",
    "team2_previous_10_average_map_score",
    "map_score_diff"
]].head()

,team1_previous_10_average_map_score,team2_previous_10_average_map_score,map_score_diff
0,13.000000,13.000000,0.0
1,12.333333,12.333333,0.0
2,11.600000,11.600000,0.0
3,11.571429,11.571429,0.0
4,10.666667,10.666667,0.0


In [14]:
numeric_features = [
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "adr_diff",
    "kast_diff",
    "kddiff_diff",
    "map_score_diff",
    "bestOf"
]

categorical_features = [
    "map_name"
]

X = df[numeric_features + categorical_features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (5539, 9)
y shape: (5539,)


,kills_diff,deaths_diff,assists_diff,adr_diff,kast_diff,kddiff_diff,map_score_diff,bestOf,map_name
0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,Overpass
1,0.6,-0.6,-3.2,0.52,1.68,1.2,0.0,3.0,Ancient
2,1.5,-1.6,-0.8,3.11,8.47,3.1,0.0,3.0,Inferno
3,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,Overpass
4,9.0,-9.0,3.0,46.80,48.00,18.0,0.0,3.0,Anubis


In [15]:
##One-hot encode categorical features

X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

print("X shape after one-hot encoding:", X.shape)

X.head()

X shape after one-hot encoding: (5539, 16)


,kills_diff,deaths_diff,assists_diff,adr_diff,kast_diff,kddiff_diff,map_score_diff,bestOf,map_name_Anubis,map_name_Dust2,map_name_Inferno,map_name_Mirage,map_name_Nuke,map_name_Overpass,map_name_Train,map_name_Vertigo
0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,False,False,False,False,False,True,False,False
1,0.6,-0.6,-3.2,0.52,1.68,1.2,0.0,3.0,False,False,False,False,False,False,False,False
2,1.5,-1.6,-0.8,3.11,8.47,3.1,0.0,3.0,False,False,True,False,False,False,False,False
3,0.0,0.0,0.0,0.00,0.00,0.0,0.0,3.0,False,False,False,False,False,True,False,False
4,9.0,-9.0,3.0,46.80,48.00,18.0,0.0,3.0,True,False,False,False,False,False,False,False


In [16]:
# Handle missing values

print("Missing values before filling:")
print(X.isna().sum().sort_values(ascending=False).head(20))

X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

print("\nMissing values after filling:")
print(X.isna().sum().sum())

Missing values before filling:
kills_diff           0
deaths_diff          0
assists_diff         0
adr_diff             0
kast_diff            0
kddiff_diff          0
map_score_diff       0
bestOf               0
map_name_Anubis      0
map_name_Dust2       0
map_name_Inferno     0
map_name_Mirage      0
map_name_Nuke        0
map_name_Overpass    0
map_name_Train       0
map_name_Vertigo     0
dtype: int64

Missing values after filling:
0


In [17]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Train size: (4431, 16)
Test size: (1108, 16)

Training target distribution:
team1_win
1    0.546378
0    0.453622
Name: proportion, dtype: float64

Testing target distribution:
team1_win
1    0.558664
0    0.441336
Name: proportion, dtype: float64


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [20]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print("Random Forest Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Random Forest Test Accuracy: 0.5523465703971119

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.38      0.43       489
           1       0.58      0.69      0.63       619

    accuracy                           0.55      1108
   macro avg       0.54      0.53      0.53      1108
weighted avg       0.54      0.55      0.54      1108


Confusion Matrix:
[[184 305]
 [191 428]]


In [22]:
##Check feature importance

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance.head(20)

,feature,importance
4,kast_diff,0.137052
3,adr_diff,0.136684
5,kddiff_diff,0.130169
6,map_score_diff,0.127527
0,kills_diff,0.123941
2,assists_diff,0.123507
1,deaths_diff,0.121464
7,bestOf,0.015959
12,map_name_Nuke,0.012795
10,map_name_Inferno,0.012774


In [23]:
numeric_features_v2 = [
    # Difference features
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "adr_diff",
    "kast_diff",
    "kddiff_diff",
    "map_score_diff",
    
    # Team 1 average features
    "team1_avg_kills",
    "team1_avg_deaths",
    "team1_avg_assists",
    "team1_avg_adr",
    "team1_avg_kast",
    "team1_avg_kddiff",
    
    # Team 2 average features
    "team2_avg_kills",
    "team2_avg_deaths",
    "team2_avg_assists",
    "team2_avg_adr",
    "team2_avg_kast",
    "team2_avg_kddiff",
    
    # Match format
    "bestOf"
]

categorical_features_v2 = [
    "map_name"
]



In [26]:
X_v2 = df[numeric_features_v2 + categorical_features_v2]
y = df[target]

X_v2 = pd.get_dummies(
    X_v2,
    columns=categorical_features_v2,
    drop_first=True
)

X_v2 = X_v2.fillna(X_v2.median(numeric_only=True))
X_v2 = X_v2.fillna(0)

print("X_v2 shape:", X_v2.shape)

X_v2 shape: (5539, 28)


In [28]:
split_index = int(len(df) * 0.8)

X_train_v2 = X_v2.iloc[:split_index]
X_test_v2 = X_v2.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train size:", X_train_v2.shape)
print("Test size:", X_test_v2.shape)

Train size: (4431, 28)
Test size: (1108, 28)


In [29]:
rf_v2 = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_v2.fit(X_train_v2, y_train)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)

In [31]:
y_pred_v2 = rf_v2.predict(X_test_v2)

print("Random Forest V2 Test Accuracy:", accuracy_score(y_test, y_pred_v2))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_v2))

Random Forest V2 Test Accuracy: 0.5315884476534296

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.31      0.37       489
           1       0.56      0.70      0.63       619

    accuracy                           0.53      1108
   macro avg       0.51      0.51      0.50      1108
weighted avg       0.52      0.53      0.51      1108


Confusion Matrix:
[[154 335]
 [184 435]]
